In [1]:
!pip install mlflow boto3 awscli optuna xgboost imbalanced-learn lightgbm

In [2]:
import setup_up
import os 
print(os.getenv("AWS_DEFAULT_REGION"))
print(bool(os.getenv("AWS_ACCESS_KEY_ID")))
print(bool(os.getenv("AWS_SECRET_ACCESS_KEY")))

eu-north-1
True
True


In [3]:
!aws sts get-caller-identity > /dev/null

In [4]:
import mlflow
import setup_up
SERVER_URL = os.environ["SERVER_URL"]

mlflow.set_tracking_uri(SERVER_URL)

In [5]:
mlflow.set_experiment('Exp 5 - ML Algos with HP Tuning')

/home/grayfog/miniconda3/envs/YTsentimentAnalyzer/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
2026/03/14 23:55:32 INFO mlflow.tracking.fluent: Experiment with name 'Exp 5 - ML Algos with HP Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/grayfog/YTSentiment/notebooks/ec2-13-51-197-16.eu-north-1.compute.amazonaws.com/142406186679015040', creation_time=1773528932949, experiment_id='142406186679015040', last_update_time=1773528932949, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning', tags={}>

In [6]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


In [8]:
df = pd.read_csv('../data/read_preprocessing.csv').dropna()
df.shape

(36662, 3)

In [9]:
df['category'] = df['category'].map({-1:2,0:0, 1:1})
df = df.dropna(subset=['category'])

ngram_range = (1,3)
max_features = 1000

# train-test split before vectorization and resampling
X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2,                                                         
                                                    random_state=42, stratify=df['category'])

# vectorize using TF-IDF
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

smote = SMOTE(random_state=42)
X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)

# function to log results into MlFlow
def mlflow_start_run(model_name,model,X_train,x_test,y_train,y_test):
    with mlflow.start_run() as run:
        #log the model type
        mlflow.set_tag('mlflow.runName', f'{model_name}_SMOTE_TFIDF_TRIGRMAS')
        mlflow.set_tag('experiment_type', 'algrorithm_comparison')

        # log algorithm as parameter
        mlflow.log_param('algo_name',model_name)

        # train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # log accuracy 
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric('accuracy', accuracy)
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
             if isinstance(metrics,dict):
                 for metric, value in metrics.items():
                     mlflow.log_metric(f'{label}_{metric}', value)    
        # log the model 
        mlflow.sklearn.log_model(model, name=f'{model_name}_model')

# optuna objective function for XGBoost 
def objective_xgboost(trial):
    n_estimators = trial.suggest_int('n_estimators',50,300)
    learning_rate = trial.suggest_float('learning_rate',1e-4,1e-1,log=True)
    max_depth = trial.suggest_int('max_depth',3,10)

    model = XGBClassifier(n_estimators=n_estimators,learning_rate=learning_rate,max_depth=max_depth)
    return accuracy_score(y_test,model.fit(X_train_vec,y_train).predict(X_test_vec))

# run optuna forXGBoost, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction='maximize')
    study.optimize(objective_xgboost,n_trials=30)

    # get the best model only params logs
    best_params = study.best_params
    best_model = XGBClassifier(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'])

    # log the best model with mlflow, passing algo_name as xgboost
    log_mlflow('XGBoost',best_model,X_train_vec,X_test_vec,y_train,y_test)

run_optuna_experiment()

[I 2026-03-14 23:56:54,461] A new study created in memory with name: no-name-c7f8231f-1a7f-4023-95d3-09fb97295525
[I 2026-03-14 23:59:15,863] Trial 0 finished with value: 0.5544797490795036 and parameters: {'n_estimators': 239, 'learning_rate': 0.0003325384241404214, 'max_depth': 6}. Best is trial 0 with value: 0.5544797490795036.
[I 2026-03-15 00:05:23,106] Trial 1 finished with value: 0.7785353879721806 and parameters: {'n_estimators': 290, 'learning_rate': 0.05898321342495518, 'max_depth': 10}. Best is trial 1 with value: 0.7785353879721806.
[I 2026-03-15 00:06:02,801] Trial 2 finished with value: 0.5404336560752762 and parameters: {'n_estimators': 59, 'learning_rate': 0.0004732542178520114, 'max_depth': 5}. Best is trial 1 with value: 0.7785353879721806.
[I 2026-03-15 00:11:08,954] Trial 3 finished with value: 0.603709259511796 and parameters: {'n_estimators': 193, 'learning_rate': 0.00013343423010635984, 'max_depth': 9}. Best is trial 1 with value: 0.7785353879721806.
[I 2026-03-1

NameError: name 'log_mlflow' is not defined